In [1]:
# Housing Price Prediction MVP (Finland)

#Dataset: Statistics Finland (StatFin) — 13mx (existing dwellings), annual, municipality-level.
#Target: Price per square meter (EUR/m²)
#Model: Ridge Regression (baseline linear model)

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
DATA_PATH = "data/mvp_housing_data.csv"

raw = pd.read_csv(
    DATA_PATH,
    sep=";",
    decimal=",",
    encoding="latin1",
    skiprows=2,
    header=None
)

raw.head()

,0,1,2,3
0,Vuosi,Kunta,Talotyyppi,Neliöhinta (EUR/m2)
1,2006,Espoo,Rivitalot,2706
2,2006,Espoo,Kerrostalot,2322
3,2006,Helsinki,Rivitalot,2827
4,2006,Helsinki,Kerrostalot,3141


In [4]:
df = raw.drop(index=0).reset_index(drop=True)
df.columns = ["Year", "Municipality", "BuildingType", "PricePerM2"]

df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["PricePerM2"] = pd.to_numeric(df["PricePerM2"], errors="coerce")

df = df.dropna(subset=["Year", "PricePerM2"]).copy()
df["Year"] = df["Year"].astype(int)

df.head(), df.shape

(   Year Municipality BuildingType  PricePerM2
 0  2006        Espoo    Rivitalot        2706
 1  2006        Espoo  Kerrostalot        2322
 2  2006     Helsinki    Rivitalot        2827
 3  2006     Helsinki  Kerrostalot        3141
 4  2006         Oulu    Rivitalot        1522,
 (190, 4))

In [5]:
df.isna().sum()

Year            0
Municipality    0
BuildingType    0
PricePerM2      0
dtype: int64

In [6]:
X = df[["Year", "Municipality", "BuildingType"]]
y = df["PricePerM2"]

X.head()

,Year,Municipality,BuildingType
0,2006,Espoo,Rivitalot
1,2006,Espoo,Kerrostalot
2,2006,Helsinki,Rivitalot
3,2006,Helsinki,Kerrostalot
4,2006,Oulu,Rivitalot


In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Municipality", "BuildingType"]),
        ("num", "passthrough", ["Year"]),
    ]
)

model = Ridge(alpha=1.0) #Linear model

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model),
])

In [8]:
from sklearn.model_selection import cross_val_score

mae_scores = cross_val_score(
    pipeline, X, y,
    cv=5,
    scoring="neg_mean_absolute_error"
)

print("CV MAE (EUR/m²):", round(-mae_scores.mean(), 2))
print("CV MAE std:", round(mae_scores.std(), 2))

CV MAE (EUR/m²): 273.77
CV MAE std: 52.29


In [9]:
rmse_scores = cross_val_score(
    pipeline, X, y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

print("CV RMSE (EUR/m²):", round(-rmse_scores.mean(), 2))
print("CV RMSE std:", round(rmse_scores.std(), 2))

CV RMSE (EUR/m²): 361.87
CV RMSE std: 90.38


In [10]:
#Train the model
pipeline.fit(X, y)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
demo_cases = pd.DataFrame([
    {"Year": 2023, "Municipality": "Helsinki", "BuildingType": "Kerrostalot"},
    {"Year": 2023, "Municipality": "Tampere", "BuildingType": "Kerrostalot"},
    {"Year": 2023, "Municipality": "Oulu", "BuildingType": "Rivitalot"},
])

demo_cases["Predicted_EUR_per_m2"] = pipeline.predict(demo_cases)
demo_cases

,Year,Municipality,BuildingType,Predicted_EUR_per_m2
0,2023,Helsinki,Kerrostalot,4572.636336
1,2023,Tampere,Kerrostalot,3440.467105
2,2023,Oulu,Rivitalot,2262.465662


In [ ]:
import matplotlib.pyplot as plt

helsinki = df[(df["Municipality"] == "Helsinki") & (df["BuildingType"] == "Kerrostalot")].copy()
helsinki = helsinki.sort_values("Year")

plt.figure()
plt.plot(
    helsinki["Year"],
    helsinki["PricePerM2"],
    marker="o",
    label="Historical data"
)

# future years for prediction 
future_years = pd.DataFrame({
    "Year": [2025, 2026],
    "Municipality": ["Helsinki", "Helsinki"],
    "BuildingType": ["Kerrostalot", "Kerrostalot"]
})

# Count prediction
future_preds = pipeline.predict(future_years)

# Draw prediction in the same chart
plt.plot(
    future_years["Year"],
    future_preds,
    marker="x",
    linestyle="--",
    label="Model prediction"
)

# add legend for clearer chart
plt.legend()

plt.title("Helsinki — Kerrostalot — Price per m² over time")
plt.xlabel("Year")
plt.ylabel("EUR/m²")
plt.show()

In [ ]:
helsinki.tail(6)